In [1]:
import os
import shutil
import numpy as np
from config import Config
from env import MySumoEnv

_NEEDED_WORKER_FILES = ("detectors.add.xml", "network.net.xml", "run.sumocfg")

def prepare_worker_dir(worker_idx=0):
    if hasattr(Config, "ensure_dirs"):
        Config.ensure_dirs()
    else:
        os.makedirs(Config.WORKERS_ROOT, exist_ok=True)
    src = Config.BASE_WORK_DIR
    dst = Config.worker_dir(worker_idx)
    os.makedirs(dst, exist_ok=True)
    for name in _NEEDED_WORKER_FILES:
        src_path = os.path.join(src, name)
        dst_path = os.path.join(dst, name)
        if not os.path.exists(src_path):
            raise FileNotFoundError(f"No utils file: {src_path}")
        shutil.copy2(src_path, dst_path)
    os.makedirs(os.path.join(dst, "dump"), exist_ok=True)
    return dst

def build_env(worker_idx=0, total_step=None, seed=42):
    rl_dir = prepare_worker_dir(worker_idx)
    answer_path = Config.answer_path_for(worker_idx)
    sumocfg_path = os.path.join(rl_dir, "run.sumocfg")

    if not os.path.exists(rl_dir):
        raise FileNotFoundError(f"No directory: {rl_dir}")
    if not os.path.exists(sumocfg_path):
        raise FileNotFoundError(f"No run.sumocfg: {sumocfg_path}")
    if not os.path.exists(answer_path):
        raise FileNotFoundError(f"No answer.csv: {answer_path}")

    if total_step is None:
        total_step = Config.TOTAL_STEP

    env = MySumoEnv(
        rl_dir=rl_dir,
        sumo_binary=Config.SUMO_BINARY,
        origin_list=Config.ORIGIN_LIST,
        destination_list=Config.DESTINATION_LIST,
        input_interval=Config.INPUT_INTERVAL,
        detector_interval=Config.DETECTOR_INTERVAL,
        num_OD=Config.NUM_OD,
        state_dim=1,
        answer_dir=answer_path,
        total_step=int(total_step),
    )

    obs, info = env.reset(seed=seed)
    return env, obs, info

In [2]:
import time
from trial_timing import reset_trial_times, append_trial_time
import os
import json
import numpy as np
from bayes_opt import BayesianOptimization
from config import Config

def evenly_binary_sequence(count: int, n: int) -> np.ndarray:
\
\
       
    count = int(np.clip(count, 0, n))
    seq = np.zeros(n, dtype=np.int32)
    if count == 0:
        return seq
    if count == n:
        seq[:] = 1
        return seq

    acc = 0
    for i in range(n):
        acc += count
        if acc >= n:
            seq[i] = 1
            acc -= n
    return seq

def counts_to_binary_action_plan(counts_block: np.ndarray, num_od: int, total_step: int, block_steps: int) -> np.ndarray:
\
\
\
       
    n_blocks = int(np.ceil(total_step / block_steps))
    action_plan = np.zeros((total_step, num_od), dtype=np.int32)

    for b in range(n_blocks):
        s = b * block_steps
        e = min((b + 1) * block_steps, total_step)
        n_local = e - s
        for od in range(num_od):
            c = int(np.rint(counts_block[b, od]))
            c = int(np.clip(c, 0, n_local))
            action_plan[s:e, od] = evenly_binary_sequence(c, n_local)

    return action_plan

def run_episode_with_plan(build_env_fn, worker_idx, action_plan, seed):
    env, obs, info = build_env_fn(worker_idx=worker_idx, total_step=action_plan.shape[0], seed=seed)
    total_reward = 0.0
    last_info = {}

    try:
        for t in range(action_plan.shape[0]):
            obs, reward, terminated, truncated, info = env.step(action_plan[t])
            total_reward += float(reward)
            last_info = info
            if terminated or truncated:
                break
    finally:
        env.close()

    trajectory = last_info.get("trajectory", [])
    return float(total_reward), trajectory

def bayes_optimize_counts_300s(
    build_env_fn,
    worker_idx,
    num_od,
    total_step,
    input_interval,
    detector_interval,
    init_points=0,
    n_iter=10,
    seed=42,
    verbose=2,
):
    block_steps = detector_interval // input_interval
    n_blocks = int(np.ceil(total_step / block_steps))

    var_meta = []
    pbounds = {}
    for b in range(n_blocks):
        s = b * block_steps
        e = min((b + 1) * block_steps, total_step)
        n_local = e - s
        for od in range(num_od):
            key = f"b{b}_od{od}"
            pbounds[key] = (0.0, float(n_local))
            var_meta.append((key, b, od, n_local))

    eval_count = {"n": 0}

    def objective(**kwargs):
        counts_block = np.zeros((n_blocks, num_od), dtype=np.float32)
        for key, b, od, n_local in var_meta:
            counts_block[b, od] = np.clip(kwargs[key], 0.0, float(n_local))

        action_plan = counts_to_binary_action_plan(
            counts_block=counts_block,
            num_od=num_od,
            total_step=total_step,
            block_steps=block_steps
        )

        score, _ = run_episode_with_plan(
            build_env_fn=build_env_fn,
            worker_idx=worker_idx,
            action_plan=action_plan,
            seed=seed + eval_count["n"],
        )
        eval_count["n"] += 1
        return score

    bo = BayesianOptimization(
        f=objective,
        pbounds=pbounds,
        random_state=seed,
        verbose=verbose,
        allow_duplicate_points=True,
    )
    bo.maximize(init_points=init_points, n_iter=n_iter)

    best_counts = np.zeros((n_blocks, num_od), dtype=np.float32)
    for key, b, od, n_local in var_meta:
        best_counts[b, od] = np.clip(bo.max["params"][key], 0.0, float(n_local))

    best_actions = counts_to_binary_action_plan(
        counts_block=best_counts,
        num_od=num_od,
        total_step=total_step,
        block_steps=block_steps
    )
    best_score = float(bo.max["target"])

    return best_actions, best_score, bo, best_counts

os.makedirs(Config.RESULT_DIR, exist_ok=True)

_trial_result_dir = RESULT_DIR if "RESULT_DIR" in globals() else Config.RESULT_DIR
reset_trial_times(_trial_result_dir)

for trial in range(5):
    _trial_t0 = time.perf_counter()
    trial_idx = trial
    seed_opt = int(100 + (trial_idx + 1))
                                                                          
    seed_eval = trial

    best_actions, best_score, bo, best_counts = bayes_optimize_counts_300s(
        build_env_fn=build_env,
        worker_idx=0,
        num_od=Config.NUM_OD,
        total_step=Config.TOTAL_STEP,
        input_interval=Config.INPUT_INTERVAL,
        detector_interval=Config.DETECTOR_INTERVAL,
        init_points=0,
        n_iter=300,
        seed=seed_opt,
        verbose=2,
    )

    print(f"[trial {trial}] Best total_reward:", best_score)
    print(f"[trial {trial}] Best action shape:", best_actions.shape)
    print(f"[trial {trial}] Best counts shape:", best_counts.shape)

    replay_reward, trajectory = run_episode_with_plan(
        build_env_fn=build_env,
        worker_idx=0,
        action_plan=best_actions,
        seed=seed_eval
    )
    print(f"[trial {trial}] Replay reward:", replay_reward)

    with open(os.path.join(Config.RESULT_DIR, f"trajectory_{trial}.json"), "w", encoding="utf-8") as f:
        json.dump(trajectory, f, ensure_ascii=False, indent=2, default=lambda o: o.tolist() if isinstance(o, np.ndarray) else o)
    append_trial_time(_trial_result_dir, trial, time.perf_counter() - _trial_t0)


|   iter    |  target   |  b0_od0   |  b0_od1   |  b0_od2   |  b0_od3   |  b1_od0   |  b1_od1   |  b1_od2   |  b1_od3   |  b2_od0   |  b2_od1   |  b2_od2   |  b2_od3   |  b3_od0   |  b3_od1   |  b3_od2   |  b3_od3   |  b4_od0   |  b4_od1   |  b4_od2   |  b4_od3   |  b5_od0   |  b5_od1   |  b5_od2   |  b5_od3   |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
ans: [11.  0.  3.  4.  5.  1. 20. 21.  4.], det: [12.  0. 22. 16.  4.  0. 42. 19.  8.], reward: -112.44444274902344
ans: [ 8. 11.  3. 14. 17.  9. 19. 30.  3.], det: [26. 39. 17. 38. 16. 17. 85. 94.  4.], reward: -1155.3333740234375
ans: [23.  4. 13. 20. 17. 11. 15. 23.  5.], det: [42. 30. 20. 55. 40. 10. 14. 58. 20.], reward: -476.8888854980469
ans: [15. 10. 10. 13. 17.

In [3]:
import os
import shutil
from config import Config


def remove_workers_root():
                                                            
    root = os.path.abspath(Config.WORKERS_ROOT)
    project_root = os.path.abspath(Config.RL_ROOT)
    if os.path.basename(root) != "workers":
        raise RuntimeError(f"Refusing to delete non-workers path: {root}")
    if os.path.commonpath([root, project_root]) != project_root:
        raise RuntimeError(f"Refusing to delete path outside experiment root: {root}")
    if os.path.isdir(root):
        shutil.rmtree(root)


remove_workers_root()
